# Homework 2B: PCA calculations and application

Work through a small PCA calculation, explain what its results mean, then apply PCA to four penguin measurements. Choose how many components to retain and inspect one example of recovering approximate measurements from them.

Use Lecture 04a as your reference. Complete the two marked code tasks and answer the questions briefly. The remaining code is supplied. You do not need an earlier homework notebook. This portion is worth 10 points and is due September 18 at 11:59 p.m. Central.

## 1. Work through the calculation

These made-up observations have two measurements, both in centimeters. For this example, center the values and keep their original scales.

| Observation | $x_1$ | $x_2$ |
| --- | ---: | ---: |
| A | 13 | 21 |
| B | 11 | 23 |
| C | 9 | 17 |
| D | 7 | 19 |

Show your arithmetic, using a calculator or explicit arithmetic expressions if helpful. You may type it in the response blocks or write it on paper and submit a separate PDF or clear JPG/PNG photographs with your notebook. If you attach handwritten work, include your name and labels 1a–1d, and write "See attached calculations" in those notebook responses. A library's covariance or PCA output alone does not show the requested steps. Fractions and radicals are welcome; otherwise, use at least three decimal places.

**1a.** Calculate the mean of each feature, then give the four centered observations.

> Replace this text with your means and centered observations for 1a.

**1b.** Calculate the two sample variances and the sample covariance. Show the sums used and place the results in the covariance matrix $S$. With four observations, divide each sum by $n-1=3$.

$$
S=\begin{bmatrix}
\operatorname{Var}(x_1) & \operatorname{Cov}(x_1,x_2)\\
\operatorname{Cov}(x_1,x_2) & \operatorname{Var}(x_2)
\end{bmatrix}
$$

> Replace this text with your calculations and matrix for 1b.

**1c.** The leading unit direction and its eigenvalue are supplied below. Multiply your matrix $S$ by $w$ and verify that the result equals $\lambda w$.

$$
w=\frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix},
\qquad \lambda=\frac{32}{3}.
$$

> Replace this text with the two vector calculations for 1c.

**1d.** Calculate observation A's score on this direction using its centered measurements. Then calculate the fraction of total variance retained by this component: divide its eigenvalue by the sum of the two original feature variances.

> Replace this text with your score and retained variance calculation for 1d.

## 2. Fit PCA and explain the results

The prepared file contains 342 penguins with complete bill length, bill depth, flipper length, and body mass measurements. It comes from the [Palmer Penguins project](https://allisonhorst.github.io/palmerpenguins/) by Horst, Hill, and Gorman, using data collected by Kristen Gorman and the Palmer Station Long Term Ecological Research program. The data are distributed under CC0.

Bill and flipper measurements are in millimeters; mass is in grams. Two incomplete records were excluded. `record_id` is a course row label, not a feature. These records describe penguins measured at particular Antarctic islands and years.

The setup subtracts each feature's mean and divides by its sample standard deviation, following 04a. It saves those means and scales.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA

DATA_URL = (
    "https://raw.githubusercontent.com/olearydj/INSY7130/"
    "main/homework/hw2b/data/penguin-measurements.csv"
)
data_candidates = [
    Path("data/penguin-measurements.csv"),
    Path("content/homework/hw2b/data/penguin-measurements.csv"),
]
data_source = next((path for path in data_candidates if path.is_file()), DATA_URL)
FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
penguins = pd.read_csv(data_source).set_index("record_id")[FEATURES]
feature_means = penguins.mean()
feature_scales = penguins.std(ddof=1)
standardized = (penguins - feature_means) / feature_scales

print("Records and features:", penguins.shape)
display(penguins.head())

**2a.** Why standardize these features before fitting PCA? Would fitting PCA to the original columns automatically perform this scaling?

> Replace this text with your answer to 2a.

Recall the fitted-object pattern from 04a:

```python
model = PCA(n_components=2, svd_solver="full")
model.fit(measurements)
scores = model.transform(measurements)
```

Adapt this pattern to keep all four components so you can inspect their variance contributions. The constructor sets the component count, `.fit()` learns from the data, and `.transform()` uses the fitted model.

Complete the task cell by fitting `pca` to `standardized` and saving its transformed scores as `penguin_scores`.

In [ ]:
pca = PCA(n_components=4, svd_solver="full")
penguin_scores = None
# TODO: Fit pca to standardized and assign its transformed scores to penguin_scores.

Following 04a, we call the direction coefficients **loadings**. The next cell displays loadings, observation scores, and retained variance. `pca.components_` has one row per component; `.T` displays components as columns in the loadings table.

In [ ]:
if penguin_scores is None:
    print("Complete the fit/transform task above to inspect the PCA results.")
else:
    component_names = ["PC1", "PC2", "PC3", "PC4"]
    loadings = pd.DataFrame(
        pca.components_.T, index=FEATURES, columns=component_names
    )
    variance_summary = pd.DataFrame(
        {
            "Eigenvalue (component variance)": pca.explained_variance_,
            "Variance retained (%)": 100 * pca.explained_variance_ratio_,
            "Cumulative retained (%)": 100 * pca.explained_variance_ratio_.cumsum(),
        },
        index=component_names,
    )
    print("Coefficient array shape:", pca.components_.shape)
    print("Score array shape:", penguin_scores.shape)
    print("Loadings (component coefficients):")
    display(loadings.round(4))
    print("Scores for the first five penguins:")
    display(pd.DataFrame(penguin_scores, index=penguins.index,
                         columns=component_names).head().round(3))
    display(variance_summary.round(2))

    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
    loadings.iloc[:, :2].plot.barh(
        ax=axes[0], color=["#245c91", "#b65313"]
    )
    axes[0].axvline(0, color="0.5", linewidth=0.8)
    axes[0].set(title="First two component recipes", xlabel="Loading (coefficient)")
    axes[0].invert_yaxis()
    axes[1].plot(
        range(1, 5), variance_summary["Cumulative retained (%)"],
        "o-", color="#245c91"
    )
    axes[1].axhline(90, color="0.5", linestyle="--", label="90% target")
    axes[1].set(
        title="Cumulative variance retained", xlabel="Components retained",
        ylabel="Percent of standardized variance", xticks=range(1, 5), ylim=(0, 105)
    )
    axes[1].legend(loc="lower right")
    plt.show()

**2b.** In Part 1, which numbers are the loadings and which is observation A's score? Explain how the loadings define a component and what the score describes.

> Replace this text with your answer to 2b.

**2c.** What does `pca.fit(standardized)` learn about the components, and what does `pca.transform(standardized)` calculate for each penguin? Explain why the loadings are shared across penguins while their scores can differ.

> Replace this text with your answer to 2c.

**2d.** What does a component's eigenvalue tell us? How is that different from an individual penguin's score?

> Replace this text with your answer to 2d.

**2e.** Describe what a high score on PC1 and on PC2 tends to indicate about a penguin's measurements. Use the sizes and signs of the loadings. One sentence per component is enough.

> Replace this text with your descriptions for 2e.

## 3. Choose components and inspect one reconstruction

For this exercise, aim to retain **at least 90% of the standardized variance** with as few components as possible. Use the table and plot from Part 2.

**3a.** How many components meet that target? State the cumulative percentage for that count and for one fewer component.

> Replace this text with your answer to 3a.

**3b.** If you keep that many components, what does each column of the reduced score table represent? Explain how those columns use the four original measurements.

> Replace this text with your answer to 3b.

Replace `None` with your selected count in the task cell.

In [ ]:
selected_k = None
# TODO: Replace None with the fewest components that reach the 90% target.

The retained scores can be converted back into approximate measurements. This is called **reconstruction**. Keeping fewer components discards some variation, so the reconstructed values can differ from the originals.

Run the supplied cell to compare record 1's original body mass with its reconstruction using two components and your selected count. The code returns the results in grams. You only need to interpret the table.

In [ ]:
def reconstruct_mass(k):
    model = PCA(n_components=k, svd_solver="full").fit(standardized)
    scores = model.transform(standardized.loc[[1]])
    rebuilt = model.inverse_transform(scores)
    mass_index = FEATURES.index("body_mass_g")
    return (
        rebuilt[0, mass_index] * feature_scales["body_mass_g"]
        + feature_means["body_mass_g"]
    )


if selected_k is None:
    print("Set selected_k above to compare the reconstructed masses.")
else:
    mass_comparison = pd.DataFrame(
        {
            "Components retained": [2, selected_k],
            "Original mass (g)": [penguins.loc[1, "body_mass_g"]] * 2,
            "Reconstructed mass (g)": [reconstruct_mass(2), reconstruct_mass(selected_k)],
        }
    )
    display(mass_comparison.round(2))

**3c.** Which reconstruction is closer to the original mass? What does this example illustrate about the tradeoff between using fewer components and preserving the original measurements? Answer in two or three sentences.

> Replace this text with your comparison for 3c.

For optional detail, see "04A: Reconstructing Measurements from PCA Scores" on the [Course Addenda and Errata page](https://auburn.instructure.com/courses/1747056/pages/course-addenda-and-errata).

## Check and submit

Submit your completed `.ipynb` notebook with the Part 2–3 answers, code, tables, figure, and assistance disclosure. Include Part 1 either as typed work in that notebook or as a separate PDF or clear JPG/PNG photographs uploaded in the same Canvas submission. Restart the runtime and run all cells in order. Keep the outputs, check that your answers match them, and confirm that all submitted pages are readable.

| Evaluation | Points |
| --- | ---: |
| Part 1: hand calculations | 3.0 |
| Part 2: PCA application, terminology, and interpretation | 4.5 |
| Part 3: component choice and one reconstruction comparison | 2.0 |
| Reproducibility and assistance disclosure | 0.5 |
| **Total** | **10.0** |

Short, accurate explanations are sufficient. You may use course materials, documentation, and LLM assistance for understanding and debugging. Your submitted calculations and explanations must be your own, and you must be able to explain the work.

**Assistance disclosure:** Identify material assistance and how you checked it, or write `None`.

> Replace this text with your disclosure.

**Optional, ungraded feedback:** Approximately how much active time did you spend, excluding breaks? Which part required the most effort, and was anything unclear?

> Optional feedback.